In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession
        .builder
        .appName("local-spark-app")
        .master("local[*]")
        .getOrCreate()
)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 19:38:32 WARN Utils: Your hostname, DESKTOP-440A2SS, resolves to a loopback address: 127.0.1.1; using 172.20.131.26 instead (on interface eth0)
26/10/13 19:38:32 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/tem/de_lrn_prcts/.venv/lib/python3.14/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/13 19:38:34 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
employees = [
    (1, "Bob", "Tiger", 12432.343, "united States", "+1 2343434 34", "43 54 5465"),
    (2, "Alice", "Smith", 35000.50, "CANADA", "+1 555 123 4567", "123 Main St"),
    (3, "John", "Doe", 28900.12, "UK", "+44 7700 900123", "10 Downing St"),
    (4, "Maria", "Gonzalez", 42000.75, "MExiCO", "+52 55 1234 5678", "Av. Reforma 123"),
    (5, "Ken", "Tanaka", 39500.90, "Japan", "+81 3 1234 5678", "Tokyo, Chuo-ku"),
    (6, "Olga", "Petrova", 31000.25, "RUSSIa", "+7 999 123 45 67", "Moscow, Tverskaya 1"),
    (7, "Liam", "Brown", 27500.80, "Australia", "+61 2 1234 5678", "Sydney, George St"),
    (8, "Emma", "Lee", 45000.60, "Singapore", "+65 6123 4567", "Orchard Road 10"),
    (9, "Diego", "Rodriguez", 33000.40, "SPain", "+34 91 123 45 67", "Madrid, Gran Via"),
    (10, "Sophie", "Martin", 38000.30, "France", "+33 1 23 45 67 89", "Paris, Champs-Élysées")
]


In [4]:
df_employees = spark.createDataFrame(
    employees,
    schema="""
    employee_id INT,
    first_name STRING,
    last_name STRING,
    salary DOUBLE,
    country STRING,
    phone_number STRING,
    address STRING
    """
)

In [5]:
df_employees.printSchema()

root
 |-- employee_id: integer (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- salary: double (nullable = true)
 |-- country: string (nullable = true)
 |-- phone_number: string (nullable = true)
 |-- address: string (nullable = true)



In [7]:
from pyspark.sql.functions import lit, concat, concat_ws, col, date_format

In [8]:
help(date_format)

Help on function date_format in module pyspark.sql.functions.builtin:

date_format(date: ColumnOrName, format: str) -> Column
    Converts a date/timestamp/string to a value of string in the format specified by the date
    format given by the second argument.

    A pattern could be for instance `dd.MM.yyyy` and could return a string like '18.03.1993'. All
    pattern letters of `datetime pattern`_. can be used.

    .. _datetime pattern: https://spark.apache.org/docs/latest/sql-ref-datetime-pattern.html

    .. versionadded:: 1.5.0

    .. versionchanged:: 3.4.0
        Supports Spark Connect.

    Notes
    -----
    Whenever possible, use specialized functions like `year`.

    Parameters
    ----------
    date : :class:`~pyspark.sql.Column` or column name
        input column of values to format.
    format: literal string
        format to use to represent datetime values.

    See Also
    --------
    :meth:`pyspark.sql.functions.to_date`
    :meth:`pyspark.sql.functions.to_ti

In [9]:
(
    df_employees
        .groupBy("country")
        .count()
        .show()
)

+-------------+-----+
|      country|count|
+-------------+-----+
|United States|    1|
|       Canada|    1|
|       Mexico|    1|
|           UK|    1|
|       Russia|    1|
|        Japan|    1|
|    Singapore|    1|
|       France|    1|
|        Spain|    1|
|    Australia|    1|
+-------------+-----+



In [13]:
df_employees.orderBy("employee_id").show()

+-----------+----------+---------+---------+-------------+-----------------+--------------------+
|employee_id|first_name|last_name|   salary|      country|     phone_number|             address|
+-----------+----------+---------+---------+-------------+-----------------+--------------------+
|          1|       Bob|    Tiger|12432.343|United States|    +1 2343434 34|          43 54 5465|
|          2|     Alice|    Smith|  35000.5|       Canada|  +1 555 123 4567|         123 Main St|
|          3|      John|      Doe| 28900.12|           UK|  +44 7700 900123|       10 Downing St|
|          4|     Maria| Gonzalez| 42000.75|       Mexico| +52 55 1234 5678|     Av. Reforma 123|
|          5|       Ken|   Tanaka|  39500.9|        Japan|  +81 3 1234 5678|      Tokyo, Chuo-ku|
|          6|      Olga|  Petrova| 31000.25|       Russia| +7 999 123 45 67| Moscow, Tverskaya 1|
|          7|      Liam|    Brown|  27500.8|    Australia|  +61 2 1234 5678|   Sydney, George St|
|          8|      E

In [15]:
from pyspark.sql.functions import upper

(
    df_employees
        .select(upper("first_name"), upper("last_name"))
        .show()
)

+-----------------+----------------+
|upper(first_name)|upper(last_name)|
+-----------------+----------------+
|              BOB|           TIGER|
|            ALICE|           SMITH|
|             JOHN|             DOE|
|            MARIA|        GONZALEZ|
|              KEN|          TANAKA|
|             OLGA|         PETROVA|
|             LIAM|           BROWN|
|             EMMA|             LEE|
|            DIEGO|       RODRIGUEZ|
|           SOPHIE|          MARTIN|
+-----------------+----------------+



In [16]:
(
    df_employees
        .select(upper(col("first_name")), upper(col("last_name")))
        .show()
)

+-----------------+----------------+
|upper(first_name)|upper(last_name)|
+-----------------+----------------+
|              BOB|           TIGER|
|            ALICE|           SMITH|
|             JOHN|             DOE|
|            MARIA|        GONZALEZ|
|              KEN|          TANAKA|
|             OLGA|         PETROVA|
|             LIAM|           BROWN|
|             EMMA|             LEE|
|            DIEGO|       RODRIGUEZ|
|           SOPHIE|          MARTIN|
+-----------------+----------------+



In [21]:
(
    df_employees
        .groupBy(upper(col("country")))
        .count()
        .show()
)

(
    df_employees
        .groupBy(upper(col("country")).alias("country"))
        .count()
        .show()
)

(
    df_employees
        .groupBy(upper(col("country")).alias("country"))
        .count()
        .orderBy(col("country").asc())
        .show()
)

+--------------+-----+
|upper(country)|count|
+--------------+-----+
|        CANADA|    1|
| UNITED STATES|    1|
|        MEXICO|    1|
|            UK|    1|
|         JAPAN|    1|
|        RUSSIA|    1|
|        FRANCE|    1|
|     SINGAPORE|    1|
|         SPAIN|    1|
|     AUSTRALIA|    1|
+--------------+-----+

+-------------+-----+
|      country|count|
+-------------+-----+
|       CANADA|    1|
|UNITED STATES|    1|
|       MEXICO|    1|
|           UK|    1|
|        JAPAN|    1|
|       RUSSIA|    1|
|       FRANCE|    1|
|    SINGAPORE|    1|
|        SPAIN|    1|
|    AUSTRALIA|    1|
+-------------+-----+

+-------------+-----+
|      country|count|
+-------------+-----+
|    AUSTRALIA|    1|
|       CANADA|    1|
|       FRANCE|    1|
|        JAPAN|    1|
|       MEXICO|    1|
|       RUSSIA|    1|
|    SINGAPORE|    1|
|        SPAIN|    1|
|           UK|    1|
|UNITED STATES|    1|
+-------------+-----+



In [23]:
df_employees.orderBy(col("employee_id").desc()).show()

+-----------+----------+---------+---------+-------------+-----------------+--------------------+
|employee_id|first_name|last_name|   salary|      country|     phone_number|             address|
+-----------+----------+---------+---------+-------------+-----------------+--------------------+
|         10|    Sophie|   Martin|  38000.3|       France|+33 1 23 45 67 89|Paris, Champs-Ély...|
|          9|     Diego|Rodriguez|  33000.4|        Spain| +34 91 123 45 67|    Madrid, Gran Via|
|          8|      Emma|      Lee|  45000.6|    Singapore|    +65 6123 4567|     Orchard Road 10|
|          7|      Liam|    Brown|  27500.8|    Australia|  +61 2 1234 5678|   Sydney, George St|
|          6|      Olga|  Petrova| 31000.25|       Russia| +7 999 123 45 67| Moscow, Tverskaya 1|
|          5|       Ken|   Tanaka|  39500.9|        Japan|  +81 3 1234 5678|      Tokyo, Chuo-ku|
|          4|     Maria| Gonzalez| 42000.75|       Mexico| +52 55 1234 5678|     Av. Reforma 123|
|          3|      J

In [26]:
df_employees. \
    orderBy(upper(df_employees.first_name)). \
    show()

+-----------+----------+---------+---------+-------------+-----------------+--------------------+
|employee_id|first_name|last_name|   salary|      country|     phone_number|             address|
+-----------+----------+---------+---------+-------------+-----------------+--------------------+
|          2|     Alice|    Smith|  35000.5|       Canada|  +1 555 123 4567|         123 Main St|
|          1|       Bob|    Tiger|12432.343|United States|    +1 2343434 34|          43 54 5465|
|          9|     Diego|Rodriguez|  33000.4|        Spain| +34 91 123 45 67|    Madrid, Gran Via|
|          8|      Emma|      Lee|  45000.6|    Singapore|    +65 6123 4567|     Orchard Road 10|
|          3|      John|      Doe| 28900.12|           UK|  +44 7700 900123|       10 Downing St|
|          5|       Ken|   Tanaka|  39500.9|        Japan|  +81 3 1234 5678|      Tokyo, Chuo-ku|
|          7|      Liam|    Brown|  27500.8|    Australia|  +61 2 1234 5678|   Sydney, George St|
|          4|     Ma

In [30]:
(
    df_employees
        .select("employee_id",
                concat("first_name", lit(" "), "last_name").alias("fullname"))
        .show()
)

+-----------+---------------+
|employee_id|       fullname|
+-----------+---------------+
|          1|      Bob Tiger|
|          2|    Alice Smith|
|          3|       John Doe|
|          4| Maria Gonzalez|
|          5|     Ken Tanaka|
|          6|   Olga Petrova|
|          7|     Liam Brown|
|          8|       Emma Lee|
|          9|Diego Rodriguez|
|         10|  Sophie Martin|
+-----------+---------------+



In [32]:
spark.stop()